# Multimodal PDF search*<br><br>

Handling PDF files, which often contain a mix of text, images, and layout information, presents a challenge for traditional embedding methods. This usually requires a multimodal generative model to pre-process the documents into a format that is suitable for the embedding model.
<br>

In [1]:
import os

from getpass import getpass
import nest_asyncio

from dotenv import load_dotenv

nest_asyncio.apply()

load_dotenv()
CO_API_KEY = os.environ['CO_API_KEY'] or getpass("Enter your Cohere API key: ")

In [2]:
import cohere
import numpy as np
co = cohere.Client(
    api_key=CO_API_KEY
)  # Get your free API key: https://dashboard.cohere.com/api-keys

In [3]:
from pdf2image import convert_from_path
from io import BytesIO
import base64
import chromadb
import cohere


ModuleNotFoundError: No module named 'pdf2image'

Embed v5 solves this problem as it is designed to natively understand mixed-modality inputs. Embed v5 can directly process the PDF content, including text and images, in a single step. It generates a unified embedding that captures the semantic meaning derived from both the textual and visual elements.>

In [3]:
### STEP 1: Embed the documents

documents = [
    "Remboursement des frais de voyage : Gérez facilement vos frais de voyage en les soumettant via notre outil financier. Les approbations sont rapides et simples.",
    "Travailler de l'étranger : Il est possible de travailler à distance depuis un autre pays. Il suffit de coordonner avec votre responsable et de vous assurer d'être disponible pendant les heures de travail.",
    "Avantages pour la santé et le bien-être : Nous nous soucions de votre bien-être et proposons des adhésions à des salles de sport, des cours de yoga sur site et une assurance santé complète.",
    "Fréquence des évaluations de performance : Nous organisons des bilans informels tous les trimestres et des évaluations formelles deux fois par an.",
]

# Embed the documents
doc_emb = co.embed(
    texts=documents,
    model="embed-v5.0-pro",
    input_type="search_document",
    embedding_types=["float"],
).embeddings.float

### STEP 2: Embed the query

# Add the user query
query = "What's your remote-working policy?"

# Embed the query
query_emb = co.embed(
    texts=[query],
    model="embed-v5.0-pro",
    input_type="search_query",
    embedding_types=["float"],
).embeddings.float

### STEP 3: Return the most similar documents

# Calculate similarity scores
scores = np.dot(query_emb, np.transpose(doc_emb))[0]

# Sort and filter documents based on scores
top_n = 4
top_doc_idxs = np.argsort(-scores)[:top_n]

# Display search results
for idx, docs_idx in enumerate(top_doc_idxs):
    print(f"Rank: {idx+1}")
    print(f"Document: {documents[docs_idx]}\n")


Rank: 1
Document: Travailler de l'étranger : Il est possible de travailler à distance depuis un autre pays. Il suffit de coordonner avec votre responsable et de vous assurer d'être disponible pendant les heures de travail.

Rank: 2
Document: Fréquence des évaluations de performance : Nous organisons des bilans informels tous les trimestres et des évaluations formelles deux fois par an.

Rank: 3
Document: Remboursement des frais de voyage : Gérez facilement vos frais de voyage en les soumettant via notre outil financier. Les approbations sont rapides et simples.

Rank: 4
Document: Avantages pour la santé et le bien-être : Nous nous soucions de votre bien-être et proposons des adhésions à des salles de sport, des cours de yoga sur site et une assurance santé complète.

